# Visualizations

Plotly charts from categorized discovery records.
Run `notebook/analysis.ipynb` first so CSVs exist under `output/processed/`.

Paths are anchored at the project root (`simple/`), one level above this notebook.


In [10]:

%pip install -q -r {_requirements()}
%pip install -q "nbformat>=4.2.0"


zsh: parse error near `}'


Python(28415) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


Python(28416) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


# 0. Imports & paths


In [11]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import umap

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots


def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "scripts" / "sentiment_analysis.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected scripts/sentiment_analysis.py). "
        f"cwd={here}"
    )


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.categorize_records import EMBEDDING_MODEL

CATEGORY_ORDER = [
    "Workflow & Business Processes",
    "Case Management",
    "Data Management & Visibility",
    "Records & Document Management",
    "System Integration",
    "Reporting & Decision Support",
    "Communication & Collaboration",
    "Scheduling & Resource Management",
    "User Experience & Performance",
    "Training & Documentation",
]

PROCESSED = ROOT / "output" / "processed"
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CATEGORIZED_CHALLENGES_CSV = PROCESSED / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED / "categorized_expectations.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


## 1. Load categorized records

Reads `categorized_challenges.csv` / `categorized_expectations.csv` written by analysis.
The semantic map cell re-embeds challenges (embeddings are not persisted).


In [12]:
missing = [p for p in (CATEGORIZED_CHALLENGES_CSV, CATEGORIZED_EXPECTATIONS_CSV) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Run notebook/analysis.ipynb through the save step first. Missing:\n  - "
        + "\n  - ".join(str(p) for p in missing)
    )

df = pd.read_csv(CATEGORIZED_CHALLENGES_CSV)
expectations_df = pd.read_csv(CATEGORIZED_EXPECTATIONS_CSV)

print(f"Challenges: {len(df)} | Expectations: {len(expectations_df)}")
print(f"Categories: {df['Category'].nunique()} | Focus groups: {df['focus_group'].nunique()}")


Challenges: 1047 | Expectations: 439
Categories: 10 | Focus groups: 20


## 2. Category counts


In [13]:
category_counts = (
    df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain point categories", color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_cat.show()

expectation_counts = (
    expectations_df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_exp_cat = px.bar(
    expectation_counts, x="Count", y="Category", orientation="h",
    title="Expectation categories", color="Count", color_continuous_scale="Teal",
)
fig_exp_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_exp_cat.show()


## 4. Challenges vs expectations by theme


In [14]:
ch_c = df["Category"].value_counts()
ex_c = expectations_df["Category"].value_counts()
cats = sorted(set(ch_c.index) | set(ex_c.index), key=lambda c: -(ch_c.get(c, 0) + ex_c.get(c, 0)))
plot_cmp = pd.DataFrame({
    "Category": cats,
    "Challenges": [int(ch_c.get(c, 0)) for c in cats],
    "Expectations": [int(ex_c.get(c, 0)) for c in cats],
}).melt(id_vars="Category", var_name="Type", value_name="Count")
fig_cmp = px.bar(
    plot_cmp, x="Count", y="Category", color="Type", barmode="group", orientation="h",
    title="Challenges vs expectations by theme",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
)
fig_cmp.update_layout(height=520, template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig_cmp.show()


## 5. Top focus groups


In [15]:
top_n = 20
top = df.groupby("focus_group").size().sort_values(ascending=False).head(top_n)
exp = expectations_df.groupby("focus_group").size()
plot_top = pd.DataFrame({
    "Focus Group": top.index.tolist(),
    "Challenges": top.values.astype(int),
    "Expectations": [int(exp.get(g, 0)) for g in top.index],
}).melt(id_vars="Focus Group", var_name="Kind", value_name="Count")
fig_top = px.bar(
    plot_top,
    x="Focus Group",
    y="Count",
    color="Kind",
    barmode="group",
    title=f"Top {top_n} focus groups",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
    category_orders={"Focus Group": top.index.tolist()},
)
fig_top.update_layout(
    height=560,
    width=1000,
    template="plotly_white",
    xaxis=dict(tickangle=-40),
    margin=dict(b=140),
)
fig_top.show()


## 6. Focus group × category heatmap


In [16]:
heatmap_df = pd.crosstab(df["focus_group"], df["Category"])
focus_group_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[focus_group_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Group", y="Category", color="Count"),
    color_continuous_scale="Teal", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(height=700, width=1100, xaxis_tickangle=-45, template="plotly_white")
fig_heatmap.show()


In [ ]:
from ipywidgets import Output, VBox, HBox, HTML, Dropdown, Button
from IPython.display import clear_output

# Ensure output dir exists even if imports cell wasn't re-run after FIG_DIR was added
FIG_DIR = Path("output/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
challenges = df  # load cell uses df; detail view expects challenges

heat = pd.crosstab(df["focus_group"], df["Category"])
heat = heat.reindex(columns=[c for c in CATEGORY_ORDER if c in heat.columns])
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]
# Swap axes: categories on Y, focus groups on X
heat = heat.T

x_labels = heat.columns.tolist()  # focus groups
y_labels = heat.index.tolist()    # categories
z = heat.values.astype(int)
text = [[("" if v == 0 else str(v)) for v in row] for row in z]

DETAIL_COLS = [
    "focus_group",
    "pain_points",
    "Category",
    "Category_Method",
    "Category_Confidence",
    "Cluster_Label",
]

heat_fig = go.FigureWidget(
    data=[
        go.Heatmap(
            z=z,
            x=x_labels,
            y=y_labels,
            text=text,
            texttemplate="%{text}",
            textfont=dict(size=11, color="#1a1a1a"),
            colorscale="YlOrRd",
            colorbar=dict(title="Count"),
            hovertemplate=(
                "<b>%{y}</b><br>%{x}<br>"
                "Count: %{z}<br>"
                "<i>Click to show records</i>"
                "<extra></extra>"
            ),
            xgap=1,
            ygap=1,
        )
    ]
)
heat_fig.update_layout(
    title="Challenge volume: category × focus group (click a cell)",
    template="plotly_white",
    font=dict(family="IBM Plex Sans, Helvetica, Arial, sans-serif", size=13),
    height=max(480, 36 * len(y_labels) + 160),
    margin=dict(l=40, r=30, t=60, b=120),
    xaxis=dict(title="Focus group", tickangle=-40, side="bottom"),
    yaxis=dict(title="Category", autorange="reversed"),
)

ALL = "All"

record_out = Output()
status = HTML(value="<i>Click a heatmap cell (or use the dropdowns) to list matching challenges.</i>")
dd_group = Dropdown(
    options=[ALL] + x_labels,
    value=ALL,
    description="Focus group:",
    layout={"width": "420px"},
)
dd_cat = Dropdown(
    options=[ALL] + y_labels,
    value=ALL,
    description="Category:",
    layout={"width": "360px"},
)
btn = Button(description="Show records", button_style="primary")


_click_state = {"t": 0.0, "key": None}


def show_cell_records(focus_group: str, category: str):
    mask = pd.Series(True, index=challenges.index)
    if focus_group != ALL:
        mask &= challenges["focus_group"] == focus_group
    if category != ALL:
        mask &= challenges["Category"] == category
    # One row per challenge — drop exact duplicates from merged sources
    subset = (
        challenges.loc[mask, DETAIL_COLS]
        .drop_duplicates()
        .sort_values("Category_Confidence", ascending=False)
    )
    status.value = (
        f"<b>{len(subset)}</b> challenges · "
        f"<b>{focus_group}</b> × <b>{category}</b>"
    )
    with record_out:
        clear_output(wait=True)
        if subset.empty:
            print("No records for this selection.")
        else:
            display(subset)


def _on_heat_click(trace, points, selector):
    import time

    if not points.point_inds:
        return
    # After swap: x = focus group, y = category
    focus_group = points.xs[0] if points.xs else None
    category = points.ys[0] if points.ys else None
    if focus_group not in x_labels or category not in y_labels:
        if (
            points.xs
            and points.ys
            and isinstance(points.ys[0], (int, float))
            and isinstance(points.xs[0], (int, float))
        ):
            category = y_labels[int(points.ys[0])]
            focus_group = x_labels[int(points.xs[0])]
        else:
            return

    # Plotly/Jupyter often fires the same heatmap click 2–3 times
    key = (focus_group, category)
    now = time.monotonic()
    if key == _click_state["key"] and (now - _click_state["t"]) < 0.5:
        return
    _click_state["key"] = key
    _click_state["t"] = now

    dd_group.value = focus_group
    dd_cat.value = category
    show_cell_records(focus_group, category)


def _on_button_click(_):
    show_cell_records(dd_group.value, dd_cat.value)


# Replace handlers so re-running this cell does not stack callbacks
heat_trace = heat_fig.data[0]
if hasattr(heat_trace, "_click_handlers"):
    heat_trace._click_handlers = ()
heat_trace.on_click(_on_heat_click)
btn.on_click(_on_button_click)

static = go.Figure(data=list(heat_fig.data), layout=heat_fig.layout)
static.write_html(FIG_DIR / "heatmap_focus_category.html", include_plotlyjs="cdn")
print(f"Saved {FIG_DIR / 'heatmap_focus_category.html'}")

display(VBox([heat_fig, HBox([dd_group, dd_cat, btn]), status, record_out]))
